In [3]:
import time
import requests
import json

API_URL = "https://www.wikidata.org/w/api.php"
USER_AGENT = "SemTabThesisBot/1.0 (https://github.com/LeiY769; contact: leiyang677@gmail.com)"
_SLEEP = 0.1
def set_rate_limit(seconds):
    global _SLEEP
    _SLEEP = max(0.0, float(seconds))
def _get(params, max_retries=5):
    headers = {"User-Agent": USER_AGENT}
    for attempt in range(max_retries):
        try:
            r = requests.get(API_URL, params={**params, "maxlag": 5},headers=headers, timeout=30)
        except requests.RequestException as e:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 429 or r.status_code >= 500:
            wait = float(r.headers.get("Retry-After", 2 ** attempt))
            time.sleep(wait)
            continue
        if r.status_code == 200:
            if _SLEEP:
                time.sleep(_SLEEP)
            return r.json()
        return None       
    raise RuntimeError("Wikidata API failed after retries") 
def search_entities(query, language="en", limit=20):
    if not query:
        return []
    data = _get({"action": "wbsearchentities","search": query, "language": language,"limit": limit,"format": "json"})
    if not data:
        return []
    return [(e.get("label", ""), e.get("id", "")) for e in data.get("search", [])]
def fulltext_search(query, language="en", limit=20):
    if not query:
        return []
    data = _get({"action": "query", "list": "search","srsearch": query,"srlimit": limit,"format": "json"})
    if not data:
        return []
    qids = [r["title"] for r in data.get("query", {}).get("search", [])
            if str(r.get("title", "")).startswith("Q")]
    labels = get_labels(qids, language)
    return [(labels.get(q, q), q) for q in qids]
def get_entity_data(qids, language="en"):
    out = {}
    qids = [q for q in dict.fromkeys(qids) if q]
    for i in range(0, len(qids), 50):
        chunk = qids[i:i + 50]
        data = _get({"action": "wbgetentities", "ids": "|".join(chunk),  "props": "descriptions|aliases|claims|sitelinks", "languages": language,   "sitefilter": f"{language}wiki",  "format": "json"})
        if not data:
            continue
        for qid, entity in data.get("entities", {}).items():
            claims = entity.get("claims", {})
            def _entity_ids(pid):
                ids = []
                for claim in claims.get(pid, []):
                    datavalue = claim.get("mainsnak", {}).get("datavalue")
                    if datavalue and datavalue.get("type") == "wikibase-entityid":
                        ids.append(datavalue["value"].get("id", ""))
                return [x for x in ids if x]

            out[qid] = {"description": entity.get("descriptions", {}).get(language, {}).get("value", ""),"aliases": [a["value"] for a in entity.get("aliases", {}).get(language, [])],"P31": _entity_ids("P31"),"P279": _entity_ids("P279"), "sitelink": entity.get("sitelinks", {}).get(f"{language}wiki", {}).get("title", "")}
    return out
def _parse_datavalue(dv):
    if not dv:
        return None
    t = dv.get("type")
    v = dv.get("value")
    if t == "wikibase-entityid":
        return ["entity", v.get("id")]
    if t == "quantity":
        try:
            return ["quantity", float(v.get("amount"))]
        except (TypeError, ValueError):
            return None
    if t == "time":
        return ["time", v.get("time")]
    if t == "string":
        return ["string", v]
    if t == "monolingualtext":
        return ["string", v.get("text")]
    if t == "globecoordinate":
        return ["coord", [v.get("latitude"), v.get("longitude")]]
    return None
def get_entity_claims(qids):
    out = {}
    qids = [q for q in dict.fromkeys(qids) if q]
    for i in range(0, len(qids), 50):
        chunk = qids[i:i + 50]
        data = _get({"action": "wbgetentities", "ids": "|".join(chunk),
                     "props": "claims", "format": "json"})
        if not data:
            continue
        for qid, entity in data.get("entities", {}).items():
            props = {}
            for pid, claims in entity.get("claims", {}).items():
                vals = []
                for claim in claims:
                    parsed = _parse_datavalue(claim.get("mainsnak", {}).get("datavalue"))
                    if parsed:
                        vals.append(parsed)
                if vals:
                    props[pid] = vals
            out[qid] = props
    return out
def get_labels(qids, language="en"):
    out = {}
    qids = [q for q in qids if q]
    for i in range(0, len(qids), 50):
        chunk = qids[i:i + 50]
        data = _get({ "action": "wbgetentities", "ids": "|".join(chunk),  "props": "labels",  "languages": f"{language}|en",  "format": "json"})
        if not data:
            continue
        for qid, entity in data.get("entities", {}).items():
            labels = entity.get("labels", {})
            lab = labels.get(language) or labels.get("en")
            out[qid] = lab["value"] if lab else qid
    return out
def get_entities(qids, language="en"):
    #Entity data plus label, fetched in batches for the whole qid list.
    qids = [q for q in dict.fromkeys(qids) if q]
    if not qids:
        return {}
    data = get_entity_data(qids, language=language)
    labels = get_labels(qids, language=language)
    out = {}
    for q in qids:
        entry = data.get(q, {})
        entry["label"] = labels.get(q, q)
        out[q] = entry
    return out

def describe_cta(result, language="en"):
    all_qids = []
    for _, p31, p279 in result:
        all_qids += list(p31) + list(p279)
    info = get_entities(all_qids, language=language)

    def expand(d):
        return [{"qid": q,"pct": pct,"label": info.get(q, {}).get("label", q),"description": info.get(q, {}).get("description", "")}for q, pct in d.items()]

    return [{"column": col_id, "P31": expand(p31), "P279": expand(p279)}for col_id, p31, p279 in result]

In [4]:
pikachu_search = search_entities("Pikachu", limit=5)
lapras_search = search_entities("Lapras", limit=5)

print("search_entities('Pikachu'):")
for label, qid in pikachu_search:
    print(f"  {qid:<12} {label}")
print("\nsearch_entities('Lapras'):")
for label, qid in lapras_search:
    print(f"  {qid:<12} {label}")

print("\nfulltext_search('Pikachu'):")
for label, qid in fulltext_search("Pikachu", limit=5):
    print(f"  {qid:<12} {label}")

PIKACHU = pikachu_search[0][1]
LAPRAS = lapras_search[0][1]
print(f"\nQIDs claimed -> Pikachu: {PIKACHU} | Lapras: {LAPRAS}")

search_entities('Pikachu'):
  Q9351        Pikachu
  Q11331388    Pikachu
  Q26775830    Pikachu
  Q126999840   Pikachu
  Q11331394    Pikachu Records

search_entities('Lapras'):
  Q2481108     Lapras
  Q65562300    Laprastegiko galtzada
  Q1092595     ready-to-assemble furniture

fulltext_search('Pikachu'):
  Q9351        Pikachu
  Q54456778    Pokémon Let's Go, Pikachu! and Let's Go, Eevee!
  Q11331388    Pikachu
  Q47492499    Detective Pikachu
  Q1988120     Pokémon Yellow

QIDs claimed -> Pikachu: Q9351 | Lapras: Q2481108


In [5]:
data = get_entity_data([PIKACHU, LAPRAS])
print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "Q9351": {
    "description": "Pokémon species, mascot of the Pokémon franchise",
    "aliases": [
      "Pika-Pika"
    ],
    "P31": [
      "Q25930719",
      "Q15141638",
      "Q386208"
    ],
    "P279": [
      "Q27301047",
      "Q1569167",
      "Q87576284",
      "Q116812940",
      "Q80447738"
    ],
    "sitelink": "Pikachu"
  },
  "Q2481108": {
    "description": "Pokémon species",
    "aliases": [
      "Laplace"
    ],
    "P31": [
      "Q25930752",
      "Q25930495"
    ],
    "P279": [
      "Q1569167"
    ],
    "sitelink": "Lapras"
  }
}


In [6]:
class_qids = []
for qid, info in data.items():
    class_qids += info["P31"] + info["P279"]
class_labels = get_labels(class_qids)

for qid, info in data.items():
    print(f"{qid}:")
    print("  P31  (instance of):", [f"{c} = {class_labels.get(c, c)}" for c in info["P31"]])
    print("  P279 (subclass of):", [f"{c} = {class_labels.get(c, c)}" for c in info["P279"]])
    print()

Q9351:
  P31  (instance of): ['Q25930719 = electric-type Pokémon', 'Q15141638 = starter Pokémon', 'Q386208 = mascot character']
  P279 (subclass of): ['Q27301047 = fictional rodent', 'Q1569167 = video game character', 'Q87576284 = manga character', 'Q116812940 = trading card game character', 'Q80447738 = anime character']

Q2481108:
  P31  (instance of): ['Q25930752 = ice-type Pokémon', 'Q25930495 = water-type Pokémon']
  P279 (subclass of): ['Q1569167 = video game character']



In [9]:

claims = get_entity_claims([PIKACHU])
pika_claims = claims[PIKACHU]
print(f"Pikachu ({PIKACHU}) owns {len(pika_claims)} properties.\n")

some_pids = list(pika_claims.keys())
pid_labels = get_labels(some_pids)
for pid in some_pids:
    values = pika_claims[pid]
    print(f"{pid:<8} {pid_labels.get(pid, pid):<40} {values}")

Pikachu (Q9351) owns 64 properties.

P31      instance of                              [['entity', 'Q25930719'], ['entity', 'Q15141638'], ['entity', 'Q386208']]
P361     part of                                  [['entity', 'Q12074899'], ['entity', 'Q857976'], ['entity', 'Q11339057'], ['entity', 'Q16656263'], ['entity', 'Q16656267'], ['entity', 'Q16656269'], ['entity', 'Q6344354'], ['entity', 'Q16656271'], ['entity', 'Q26040377'], ['entity', 'Q26040408'], ['entity', 'Q236209']]
P373     Commons category                         [['string', 'Pikachu']]
P646     Freebase ID                              [['string', '/m/01nrwb']]
P345     IMDb ID                                  [['string', 'ch0008434']]
P1080    from narrative universe                  [['entity', 'Q17562848']]
P1685    Pokémon index                            [['string', 'R-023'], ['string', 'R-002'], ['string', 'R-005'], ['string', 'N-154'], ['string', '025'], ['string', '156'], ['string', '163'], ['string', '022'], ['str